In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./fake-news/train.csv")

In [3]:
df.dropna(inplace= True)

In [4]:
import tensorflow as tf

In [5]:
tf.__version__

'2.9.0'

In [6]:
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot

In [7]:
# vocab size
voc_size = 5000

In [8]:
X = df.drop('label', axis= 1)
y = df['label']

# Onehot Representation

In [9]:
messages = X.copy()
messages.reset_index(inplace= True)

In [11]:
import nltk
import re
from nltk.corpus import stopwords

In [12]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nipuntewari/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# Data preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [15]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [16]:
one_hot_rep = [one_hot(sentence, voc_size) for sentence in corpus]

In [17]:
one_hot_rep

[[3662, 1132, 1393, 3108, 4379, 4067, 762, 1429, 2438, 1336],
 [1181, 4429, 670, 588, 970, 3437, 4393],
 [4185, 1535, 1563, 3972],
 [516, 3439, 4309, 3995, 2606, 1179],
 [3823, 970, 4112, 1480, 4584, 544, 970, 2906, 3187, 1716],
 [4009,
  1996,
  1302,
  4702,
  4334,
  2965,
  2152,
  4416,
  866,
  78,
  4956,
  3351,
  2550,
  270,
  4393],
 [2643, 3299, 3041, 1472, 1455, 3219, 2501, 2684, 4893, 808, 3190],
 [2083, 4429, 3873, 4821, 3163, 1718, 2965, 674, 4893, 808, 3190],
 [2249, 1229, 4238, 1452, 2969, 925, 287, 579, 2965, 2448],
 [1197, 3068, 1753, 1899, 3970, 4402, 609, 1446],
 [3590, 3125, 625, 2002, 4752, 3821, 4069, 1885, 1202, 4802, 4400],
 [3995, 175, 4379, 925, 2965, 3163],
 [1105, 4108, 462, 1584, 251, 4542, 4219, 14, 2690],
 [1126, 133, 2482, 305, 2258, 3013, 1709, 4893, 808, 3190],
 [2685, 899, 1335, 4307, 1930, 4893, 808, 3190],
 [3640, 4133, 3159, 2531, 4977, 4156, 71, 1864, 3998, 2646],
 [4393, 1732, 4429],
 [2851, 2995, 68, 2034, 2965, 1623, 566, 4393],
 [4994, 4033

# Embedding Representation

In [18]:
sent_length = 20
embedded_docs = pad_sequences(one_hot_rep, padding= 'pre', maxlen= sent_length)
print(embedded_docs)

[[   0    0    0 ... 1429 2438 1336]
 [   0    0    0 ...  970 3437 4393]
 [   0    0    0 ... 1535 1563 3972]
 ...
 [   0    0    0 ... 4893  808 3190]
 [   0    0    0 ... 1911  270  321]
 [   0    0    0 ... 3767   70  510]]


In [19]:
# creating model
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_length= sent_length))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss= 'binary_crossentropy', optimizer= 'adam', metrics= ['accuracy'])
print(model.summary())

2022-05-30 13:42:35.313144: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-05-30 13:42:35.314521: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Metal device set to: Apple M1

systemMemory: 8.00 GB
maxCacheSize: 2.67 GB

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 20, 40)            200000    
                                                                 
 lstm (LSTM)                 (None, 100)               56400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 256,501
Trainable params: 256,501
Non-trainable params: 0
_________________________________________________________________
None


In [20]:
import numpy as np
X_final = np.array(embedded_docs)
y_final = np.array(y)

In [21]:
X_final.shape, y_final.shape

((18285, 20), (18285,))

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size= 0.3, random_state= 23)

# Model Training

In [23]:
model.fit(
    X_train, 
    y_train, 
    validation_data= (X_test, y_test),
    epochs= 10,
    batch_size= 64
)

Epoch 1/10


2022-05-30 13:46:17.722417: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2022-05-30 13:46:18.602142: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:46:18.907508: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:46:21.268742: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


200/200 [==============================] - ETA: 0s - loss: 0.3328 - accuracy: 0.8419

2022-05-30 13:46:26.333494: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:46:26.382478: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


200/200 [==============================] - 9s 23ms/step - loss: 0.3328 - accuracy: 0.8419 - val_loss: 0.2008 - val_accuracy: 0.9123
Epoch 2/10
200/200 [==============================] - 4s 19ms/step - loss: 0.1378 - accuracy: 0.9443 - val_loss: 0.1970 - val_accuracy: 0.9174
Epoch 3/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0917 - accuracy: 0.9669 - val_loss: 0.2190 - val_accuracy: 0.9156
Epoch 4/10
200/200 [==============================] - 4s 20ms/step - loss: 0.0548 - accuracy: 0.9818 - val_loss: 0.2730 - val_accuracy: 0.9070
Epoch 5/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0365 - accuracy: 0.9891 - val_loss: 0.3175 - val_accuracy: 0.9107
Epoch 6/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0194 - accuracy: 0.9950 - val_loss: 0.3868 - val_accuracy: 0.9114
Epoch 7/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0130 - accuracy: 0.9959 - val_loss: 0.4907 - val_accuracy: 0.9089
Epoch 8/10

# Performance Metrics and Accuracy

In [36]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

172/172 [==============================] - 1s 6ms/step


In [37]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [38]:
print(confusion_matrix(y_true= y_test, y_pred= y_pred))

[[2781  317]
 [ 221 2167]]


In [39]:
print(accuracy_score(y_true= y_test, y_pred= y_pred))

0.9019321910317171


In [40]:
print(classification_report(y_true= y_test, y_pred= y_pred))

              precision    recall  f1-score   support

           0       0.93      0.90      0.91      3098
           1       0.87      0.91      0.89      2388

    accuracy                           0.90      5486
   macro avg       0.90      0.90      0.90      5486
weighted avg       0.90      0.90      0.90      5486



# Adding Dropout

In [41]:
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_length= sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss= 'binary_crossentropy', optimizer= 'adam', metrics= ['accuracy'])
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 20, 40)            200000    
                                                                 
 dropout (Dropout)           (None, 20, 40)            0         
                                                                 
 lstm_1 (LSTM)               (None, 100)               56400     
                                                                 
 dense_1 (Dense)             (None, 1)                 101       
                                                                 
Total params: 256,501
Trainable params: 256,501
Non-trainable params: 0
_________________________________________________________________
None


In [42]:
model.fit(
    X_train, 
    y_train, 
    validation_data= (X_test, y_test),
    epochs= 10,
    batch_size= 64
)

Epoch 1/10


2022-05-30 13:53:43.556575: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:53:43.810596: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:53:43.923289: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


200/200 [==============================] - ETA: 0s - loss: 0.3346 - accuracy: 0.8386

2022-05-30 13:53:47.969877: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:53:48.021431: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


200/200 [==============================] - 6s 23ms/step - loss: 0.3346 - accuracy: 0.8386 - val_loss: 0.2102 - val_accuracy: 0.9050
Epoch 2/10
200/200 [==============================] - 4s 19ms/step - loss: 0.1522 - accuracy: 0.9391 - val_loss: 0.2073 - val_accuracy: 0.9101
Epoch 3/10
200/200 [==============================] - 4s 19ms/step - loss: 0.1094 - accuracy: 0.9591 - val_loss: 0.2199 - val_accuracy: 0.9092
Epoch 4/10
200/200 [==============================] - 4s 20ms/step - loss: 0.0770 - accuracy: 0.9737 - val_loss: 0.2443 - val_accuracy: 0.9176
Epoch 5/10
200/200 [==============================] - 4s 20ms/step - loss: 0.0536 - accuracy: 0.9822 - val_loss: 0.2871 - val_accuracy: 0.9151
Epoch 6/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0399 - accuracy: 0.9866 - val_loss: 0.2920 - val_accuracy: 0.9163
Epoch 7/10
200/200 [==============================] - 4s 19ms/step - loss: 0.0297 - accuracy: 0.9913 - val_loss: 0.3769 - val_accuracy: 0.9121
Epoch 8/10

# Performance Metrics and Accuracy

In [43]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

 10/172 [>.............................] - ETA: 0s 

2022-05-30 13:54:23.471122: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2022-05-30 13:54:23.511673: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


172/172 [==============================] - 1s 6ms/step


In [44]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [45]:
print(confusion_matrix(y_true= y_test, y_pred= y_pred))

[[2819  279]
 [ 215 2173]]


In [46]:
print(accuracy_score(y_true= y_test, y_pred= y_pred))

0.909952606635071


In [47]:
print(classification_report(y_true= y_test, y_pred= y_pred))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      3098
           1       0.89      0.91      0.90      2388

    accuracy                           0.91      5486
   macro avg       0.91      0.91      0.91      5486
weighted avg       0.91      0.91      0.91      5486

